## Audit Bronze 作成ハンズオン（dirty CSV版）

このノートブックは `audit_dirty.csv` を Bronze テーブル化する演習です。

進め方:
- まずは **All Run** して、dirty CSV の影響（カラムずれ・JSON不整合）が残ることを確認
- その後、**AIチャット（Genie）** に相談しながら加工ロジックを改善
- audit では「加工ロジック以外」は提供済みです（usageで実装量を増やします）


In [ ]:
%run ../../config


### 1) Bronzeテーブル定義（先に作成）
- Bronzeは「生データを保持する層」なので、基本はSTRING中心
- 監査列 `_datasource`, `_ingest_timestamp` を追加


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

BRONZE_AUDIT_SCHEMA_SQL = """
    `event_id` STRING,
    `event_time` STRING,
    `action_name` STRING,
    `user` STRING,
    `request_params` STRING,
    `resource_name` STRING,
    `source_ip` STRING
"""

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {bronze_audit_table_path} (
    {BRONZE_AUDIT_SCHEMA_SQL},
    _datasource STRING,
    _ingest_timestamp TIMESTAMP
)
USING DELTA
""")

print("created/exists:", bronze_audit_table_path)


### 2) まずは加工なしで読み込む（All Run で現象確認）
ここでは意図的に最小オプションで読み込み、dirtyデータ由来の問題を観察します。


In [ ]:
naive_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(audit_csv_path)
)

naive_df = (
    naive_df.select("*", "_metadata")
    .withColumn("_datasource", F.col("_metadata.file_path"))
    .withColumn("_ingest_timestamp", F.from_utc_timestamp(F.col("_metadata.file_modification_time"), "Asia/Tokyo"))
    .drop("_metadata")
)

display(naive_df.limit(20))
print("naive row count:", naive_df.count())


### 3) 問題の見える化（チェック用）
- `user`, `request_params` がJSONとして読めない行
- `event_id` のフォーマット不正行
- 必須列null行

この件数が多い場合、Silverでの手戻りが増えるため Bronzeで最低限の補正を検討します。


In [ ]:
user_schema = T.StructType([
    T.StructField("email", T.StringType(), True),
    T.StructField("name", T.StringType(), True),
])
request_schema = T.StructType([
    T.StructField("full_name_arg", T.StringType(), True),
])

quality_df = (
    naive_df
    .withColumn("user_obj", F.from_json(F.col("user"), user_schema))
    .withColumn("request_obj", F.from_json(F.col("request_params"), request_schema))
    .withColumn("is_event_id_invalid", ~F.col("event_id").rlike(r"^evt_\d+$"))
    .withColumn("is_required_null", F.col("event_id").isNull() | F.col("event_time").isNull() | F.col("action_name").isNull())
    .withColumn("is_user_json_invalid", F.col("user").isNotNull() & F.col("user_obj").isNull())
    .withColumn("is_request_json_invalid", F.col("request_params").isNotNull() & F.col("request_obj").isNull())
)

summary = quality_df.select(
    F.count("*").alias("row_count"),
    F.sum(F.col("is_event_id_invalid").cast("int")).alias("invalid_event_id"),
    F.sum(F.col("is_required_null").cast("int")).alias("required_null_rows"),
    F.sum(F.col("is_user_json_invalid").cast("int")).alias("invalid_user_json"),
    F.sum(F.col("is_request_json_invalid").cast("int")).alias("invalid_request_json"),
)

display(summary)


### 4) AIチャット（Genie）で加工ロジックを作る
次セルの `bronze_ready_df` 作成部分を、以下のようなプロンプトで改善してください。

プロンプト例:
```text
DatabricksのPySparkでaudit dirty CSVをBronze化したいです。
要件:
- event_id,event_time,action_name,user,request_params,resource_name,source_ip を保持
- 先頭空白を除去（action_name, resource_name, user, request_params）
- event_idが evt_数字 形式でない行は除外
- user/request_params がJSONとして壊れている行は除外
- event_time/action_name/user/request_params がnullの行は除外
- _datasource,_ingest_timestamp は維持
- 最後に重複(event_id)を除去
この条件を満たすPySparkコードを生成してください。
```

補足:
- ここでの加工は「最低限の品質確保」が目的
- 高度な正規化は Silver で実施


In [ ]:
# TODO: ここを受講者が改善する（初期状態は naively pass-through）
# 例: trim / null除去 / JSON妥当性チェック / event_id形式チェック / 重複除去 など

bronze_ready_df = naive_df

# 受講者実装例（コメント解除して調整）
# bronze_ready_df = (
#     naive_df
#     .withColumn("action_name", F.ltrim(F.col("action_name")))
#     .withColumn("resource_name", F.ltrim(F.col("resource_name")))
#     .dropna(subset=["event_id", "event_time", "action_name", "user", "request_params"])
#     .dropDuplicates(["event_id"])
# )

print("bronze_ready row count:", bronze_ready_df.count())
display(bronze_ready_df.limit(20))


### 5) Bronzeテーブルへ書き込み


In [ ]:
(
    bronze_ready_df.write.format("delta")
    .mode("append")
    .saveAsTable(bronze_audit_table_path)
)

print("saved:", bronze_audit_table_path)


In [ ]:
display(spark.table(bronze_audit_table_path).orderBy(F.col("_ingest_timestamp").desc()).limit(50))


### 次のステップ
- この後は Silver で JSON展開・型変換・厳密な品質補正を実施
- usage は audit より受講者実装を増やす想定
